In [ ]:
import pandas as pd
import torch
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image
from sklearn.model_selection import train_test_split
import os

# 1. Comprobar la potencia de fuego
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Dispositivo de entrenamiento: {device}")
if torch.cuda.is_available():
    print(f"Tarjeta gráfica detectada: {torch.cuda.get_device_name(0)}")

# 2. Cargar el dataset preparado
df = pd.read_csv('../data/HAM10000_metadata_prepared.csv')

# Convertir las clases (texto) a números (0 a 6) para que la red neuronal lo entienda
clases_unicas = df['dx'].unique()
label_map = {clase: idx for idx, clase in enumerate(clases_unicas)}
df['label'] = df['dx'].map(label_map)

# 3. Dividir los datos: 80% Entrenamiento, 20% Validación
# stratify=df['label'] asegura que se mantenga la misma proporción de clases en ambos grupos
train_df, val_df = train_test_split(df, test_size=0.2, random_state=42, stratify=df['label'])
print(f"Imágenes para entrenar: {len(train_df)}")
print(f"Imágenes para validar: {len(val_df)}")

# 4. Clase personalizada de PyTorch para leer las fotos una a una sin colapsar la RAM
class SkinCancerDataset(Dataset):
    def __init__(self, dataframe, transform=None):
        self.dataframe = dataframe.reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.dataframe)

    def __getitem__(self, idx):
        img_path = self.dataframe.loc[idx, 'image_path']
        image = Image.open(img_path).convert('RGB')
        label = self.dataframe.loc[idx, 'label']
        
        if self.transform:
            image = self.transform(image)
            
        return image, label

# 5. Transformaciones y Data Augmentation (Imprescindible para el desbalanceo)
train_transform = transforms.Compose([
    transforms.Resize((224, 224)),       # Tamaño estándar para redes convolucionales
    transforms.RandomHorizontalFlip(),   # Voltea la imagen al azar
    transforms.RandomVerticalFlip(),     # Voltea verticalmente
    transforms.RandomRotation(20),       # Gira un poco la imagen
    transforms.ToTensor(),               # Convierte a Tensor
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]) # Normalización estándar
])

val_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# 6. Crear los DataLoaders (los "camiones" que llevan lotes de imágenes a la GPU)
train_dataset = SkinCancerDataset(train_df, transform=train_transform)
val_dataset = SkinCancerDataset(val_df, transform=val_transform)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False)

# 7. Prueba de fuego: Extraer el primer lote
images, labels = next(iter(train_loader))
print(f"\nForma del lote de imágenes: {images.shape} -> (Batch Size, Canales, Alto, Ancho)")
print(f"Forma del lote de etiquetas: {labels.shape}")